In [2]:
! pip install stable-baselines3[extra]



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [1]:

import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.distributions import Categorical
from torch.utils.data import BatchSampler, SubsetRandomSampler
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

In [2]:
NUM_EPISODES = 2000    
MAX_STEPS = 500          
GAMMA = 0.99
LAMBDA = 0.95
CLIP_EPS = 0.2
LR = 0.001                
UPDATE_EPOCHS = 10        
BATCH_SIZE = 64
UPDATE_TIMESTEP = 1000    
LOG_INTERVAL = 20
RENDER_INTERVAL = 200  

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.action_head = nn.Linear(128, action_dim)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return torch.softmax(self.action_head(x), dim=-1)

class ValueNetwork(nn.Module):
    def __init__(self, state_dim):
        super(ValueNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.value_head = nn.Linear(128, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.value_head(x)


In [5]:
class PPOAgent:
    def __init__(self, state_dim, action_dim, lr, gamma, lambda_, clip_eps):
        self.gamma = gamma
        self.lambda_ = lambda_
        self.clip_eps = clip_eps
        
        self.policy_net = PolicyNetwork(state_dim, action_dim).to(device)
        self.value_net = ValueNetwork(state_dim).to(device)
        self.policy_optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.value_optimizer = optim.Adam(self.value_net.parameters(), lr=lr)
        self.memory = []
    
    def select_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action_probs = self.policy_net(state)
        m = Categorical(action_probs)
        action = m.sample()
        return action.item(), m.log_prob(action).item()
    
    def store_transition(self, transition):
        self.memory.append(transition)
    
    def compute_advantages(self, rewards, dones, values, next_value):
        advantages = []
        gae = 0
        values = values + [next_value]
        for step in reversed(range(len(rewards))):
            delta = rewards[step] + self.gamma * values[step + 1] * (1 - dones[step]) - values[step]
            gae = delta + self.gamma * self.lambda_ * (1 - dones[step]) * gae
            advantages.insert(0, gae)
        returns = [adv + val for adv, val in zip(advantages, values[:-1])]
        advantages = torch.tensor(advantages, dtype=torch.float32).to(device)
        returns = torch.tensor(returns, dtype=torch.float32).to(device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return advantages, returns
    
    def update(self):
        if not self.memory: return
        
        # 解析数据
        states, actions, log_probs, rewards, dones = zip(*self.memory)
        states = torch.FloatTensor(np.array(states)).to(device)
        actions = torch.LongTensor(actions).to(device)
        old_log_probs = torch.FloatTensor(log_probs).to(device).detach()
        dones = list(dones)
        
        with torch.no_grad():
            values = self.value_net(states).squeeze().cpu().tolist()
            if not isinstance(values, list): values = [values]
            next_value = self.value_net(torch.FloatTensor(states[-1]).unsqueeze(0).to(device)).item()

        advantages, returns = self.compute_advantages(rewards, dones, values, next_value)
        for _ in range(UPDATE_EPOCHS):
            sampler = BatchSampler(SubsetRandomSampler(range(len(self.memory))), BATCH_SIZE, drop_last=False)
            
            for index in sampler:
                sampled_states = states[index]
                sampled_actions = actions[index]
                sampled_old_log_probs = old_log_probs[index]
                sampled_advantages = advantages[index]
                sampled_returns = returns[index]
                
                action_probs = self.policy_net(sampled_states)
                m = Categorical(action_probs)
                new_log_probs = m.log_prob(sampled_actions)
                ratios = torch.exp(new_log_probs - sampled_old_log_probs)
                
                surr1 = ratios * sampled_advantages
                surr2 = torch.clamp(ratios, 1 - self.clip_eps, 1 + self.clip_eps) * sampled_advantages
                policy_loss = -torch.min(surr1, surr2).mean()
                
                value_preds = self.value_net(sampled_states).squeeze()
                value_loss = nn.MSELoss()(value_preds, sampled_returns)
                
                self.policy_optimizer.zero_grad()
                policy_loss.backward()
                self.policy_optimizer.step()
                
                self.value_optimizer.zero_grad()
                value_loss.backward()
                self.value_optimizer.step()
        
        self.memory = []

In [6]:
def display_video(frames):
    plt.figure(figsize=(6, 4))
    patch = plt.imshow(frames[0])
    plt.axis('off')
    def animate(i):
        patch.set_data(frames[i])
    anim = animation.FuncAnimation(plt.gcf(), animate, frames=len(frames), interval=50)
    plt.close()
    return HTML(anim.to_jshtml())

def test_and_render_kaggle(agent, episode_idx):
    print(f"\nGenerating Animation for Episode {episode_idx}")
    env_vis = gym.make('CartPole-v1', render_mode='rgb_array')
    state, _ = env_vis.reset()
    done = False
    frames = []
    while not done:
        frames.append(env_vis.render())
        action, _ = agent.select_action(state)
        state, _, terminated, truncated, _ = env_vis.step(action)
        done = terminated or truncated
    env_vis.close()
    if len(frames) > 0:
        display(display_video(frames))

In [7]:
def train():
    env = gym.make('CartPole-v1')
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    agent = PPOAgent(state_dim, action_dim, LR, GAMMA, LAMBDA, CLIP_EPS)
    
    log_rewards = []
    time_step = 0  

    print(f"Start Training on {device}...")

    for episode in range(1, NUM_EPISODES + 1):
        state, _ = env.reset()
        done = False
        ep_reward = 0
        
        while not done:
            time_step += 1
            action, log_prob = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            
            agent.store_transition((state, action, log_prob, reward, done))
            state = next_state
            ep_reward += reward
            
            if time_step % UPDATE_TIMESTEP == 0:
                agent.update()
        
        log_rewards.append(ep_reward)
        
        if episode % LOG_INTERVAL == 0:
            avg_reward = sum(log_rewards) / len(log_rewards)
            print(f"Episode {episode:4d} | Avg Reward: {avg_reward:6.2f}")
            log_rewards = []
            
            if avg_reward >= 450:
                print("solved!")
                test_and_render_kaggle(agent, episode)
                break

        if episode % RENDER_INTERVAL == 0:
            test_and_render_kaggle(agent, episode)

    env.close()


In [ ]:
if __name__ == "__main__":
    train()

Start Training on cpu...
Episode   20 | Avg Reward:  17.05
Episode   40 | Avg Reward:  24.25
Episode   60 | Avg Reward:  27.45
Episode   80 | Avg Reward:  26.35
Episode  100 | Avg Reward:  49.30
Episode  120 | Avg Reward: 142.45
Episode  140 | Avg Reward: 376.35
Episode  160 | Avg Reward: 459.25
solved!

Generating Animation for Episode 160


Discrete(2)